# External data

Loads and cleans the five external datasets and merges them into one country-year panel:

- World Bank oil, coal and natural gas rents (% of GDP)
- OWID share of primary energy from fossil fuels
- OWID fossil-fuel production (TWh)

**Output** (in `datasets/prepared_data/`): `fossil_rents_by_country_year.csv`,
`energy_mix_fossil_share_by_country_year.csv`, `fossil_fuel_production_by_country_year.csv` and
`master_panel_country_year.csv`.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

Below we find the path to the datasets

In [2]:
RAW_DATA_DIR = Path("../datasets/Raw Data")
OUTPUT_DIR = Path("../datasets/prepared_data")
OUTPUT_DIR.mkdir(exist_ok=True)

For fossil fuels we have 3 different datasets for each type of fossil fuel coal, oil and gas, the datasets provide info on the major exporters of fossil fuel. We load all 3 and merge into 1 df. after which we strip out the non countries. the data set also holds regions and parts of the world so we strip those out.

In [3]:
# World Bank files have 4 header rows, then one column per year (wide format).
# We reshape each one to long format: one row per country-year.

def load_wb_indicator(path, value_name):
    df = pd.read_csv(path, skiprows=4)
    id_cols = ["Country Name", "Country Code", "Indicator Name", "Indicator Code"]
    year_cols = [c for c in df.columns if c.isdigit()]
    df = df.melt(id_vars=id_cols, value_vars=year_cols, var_name="Year", value_name=value_name)
    df["Year"] = df["Year"].astype(int)
    df = df.rename(columns={"Country Name": "country", "Country Code": "iso3"})
    return df[["iso3", "country", "Year", value_name]]

df_oil = load_wb_indicator(RAW_DATA_DIR / "API_NY.GDP.PETR.RT.ZS_DS2_en_csv_v2_40227" / "API_NY.GDP.PETR.RT.ZS_DS2_en_csv_v2_40227.csv", "oil_rents_pct_gdp")
df_coal = load_wb_indicator(RAW_DATA_DIR / "API_NY.GDP.COAL.RT.ZS_DS2_en_csv_v2_368101" / "API_NY.GDP.COAL.RT.ZS_DS2_en_csv_v2_368101.csv", "coal_rents_pct_gdp")
df_gas = load_wb_indicator(RAW_DATA_DIR / "API_NY.GDP.NGAS.RT.ZS_DS2_en_csv_v2_35861" / "API_NY.GDP.NGAS.RT.ZS_DS2_en_csv_v2_35861.csv", "gas_rents_pct_gdp")

df_rents = df_oil.merge(df_coal, on=["iso3", "country", "Year"], how="outer")
df_rents = df_rents.merge(df_gas, on=["iso3", "country", "Year"], how="outer")

# The World Bank file also contains regional/income aggregates (e.g. "World",
# "Arab World"), which we don't want. The metadata file has an empty Region
# for these, so we can use it to keep only real countries.
df_country_codes = pd.read_csv(RAW_DATA_DIR / "API_NY.GDP.PETR.RT.ZS_DS2_en_csv_v2_40227" / "Metadata_Country_API_NY.GDP.PETR.RT.ZS_DS2_en_csv_v2_40227.csv")
real_countries = df_country_codes.loc[df_country_codes["Region"].notna(), "Country Code"]
df_rents = df_rents[df_rents["iso3"].isin(real_countries)]

# Sum the three rent types into one "total fossil-fuel rents" column.
# min_count=1 makes sure a country-year with all three missing stays NaN,
# instead of silently becoming 0.
df_rents["total_fossil_rents_pct_gdp"] = df_rents[["oil_rents_pct_gdp", "coal_rents_pct_gdp", "gas_rents_pct_gdp"]].sum(axis=1, min_count=1)
df_rents = df_rents.sort_values(["iso3", "Year"])

df_rents

,iso3,country,Year,oil_rents_pct_gdp,coal_rents_pct_gdp,gas_rents_pct_gdp,total_fossil_rents_pct_gdp
0,ABW,Aruba,1960,NaN,NaN,NaN,NaN
265,ABW,Aruba,1961,NaN,NaN,NaN,NaN
530,ABW,Aruba,1962,NaN,NaN,NaN,NaN
795,ABW,Aruba,1963,NaN,NaN,NaN,NaN
1060,ABW,Aruba,1964,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...
16429,ZWE,Zimbabwe,2021,0.047769,0.290131,0.0,0.3379
16694,ZWE,Zimbabwe,2022,NaN,NaN,NaN,NaN
16959,ZWE,Zimbabwe,2023,NaN,NaN,NaN,NaN
17224,ZWE,Zimbabwe,2024,NaN,NaN,NaN,NaN


Litlle test to see data

In [4]:
df_rents.loc[(df_rents['country'] == 'Netherlands') & (df_rents['Year'] >= 2000)]

,iso3,country,Year,oil_rents_pct_gdp,coal_rents_pct_gdp,gas_rents_pct_gdp,total_fossil_rents_pct_gdp
10775,NLD,Netherlands,2000,0.042866,0.0,0.343533,0.386399
11040,NLD,Netherlands,2001,0.028884,0.0,0.747341,0.776225
11305,NLD,Netherlands,2002,0.044797,0.0,0.505351,0.550148
11570,NLD,Netherlands,2003,0.044441,0.0,0.430353,0.474794
11835,NLD,Netherlands,2004,0.048201,0.0,0.395244,0.443446
12100,NLD,Netherlands,2005,0.048689,0.0,0.294878,0.343567
12365,NLD,Netherlands,2006,0.047430,0.0,0.607060,0.654490
12630,NLD,Netherlands,2007,0.069582,0.0,0.521203,0.590786
12895,NLD,Netherlands,2008,0.076700,0.0,0.963753,1.040454
13160,NLD,Netherlands,2009,0.034232,0.0,0.686149,0.720382


The OIWD dataset on share of primary energy from fossil fuels, this shows how dependent a country is on fossil fuels. We load the data set and then rename column names to match the other dataset and remove non countries again. This dataset is limited in how many countries are included it only has 79 not the entire world.

In [5]:
df_energy_mix = pd.read_csv(RAW_DATA_DIR / "energy-mix" / "share-of-primary-energy-from-fossil-fuels.csv")
df_energy_mix = df_energy_mix.rename(columns={"Entity": "country", "Code": "iso3", "Fossil fuels": "fossil_share_pct"})

# OWID rows include continents/regions/World, which use codes like "OWID_AFR"
# instead of a real 3-letter ISO code, so this filter keeps only countries.
df_energy_mix = df_energy_mix[df_energy_mix["iso3"].str.match(r"^[A-Z]{3}$", na=False)]

df_energy_mix

,country,iso3,Year,fossil_share_pct
122,Algeria,DZA,1965,98.336790
123,Algeria,DZA,1966,98.781654
124,Algeria,DZA,1967,98.506100
125,Algeria,DZA,1968,98.074760
126,Algeria,DZA,1969,98.900490
...,...,...,...,...
4831,Vietnam,VNM,2021,88.933250
4832,Vietnam,VNM,2022,87.330260
4833,Vietnam,VNM,2023,89.237280
4834,Vietnam,VNM,2024,89.074486


Litlle test

In [6]:
df_energy_mix.loc[(df_energy_mix['country'] == 'Netherlands') & (df_energy_mix['Year'] >= 2000)]

,country,iso3,Year,fossil_share_pct
2894,Netherlands,NLD,2000,98.187130
2895,Netherlands,NLD,2001,98.137650
2896,Netherlands,NLD,2002,98.006590
2897,Netherlands,NLD,2003,98.081184
2898,Netherlands,NLD,2004,97.922410
2899,Netherlands,NLD,2005,97.346090
2900,Netherlands,NLD,2006,97.390390
2901,Netherlands,NLD,2007,97.170900
2902,Netherlands,NLD,2008,96.833430
2903,Netherlands,NLD,2009,96.335045


Annual coal, oil and gas production per country, in TWh. We apply the same renaming and aggregate filter as above, and
add total fossil-fuel production as the sum of the three fuels (`min_count=1`, so a country-year with no data at all
stays missing instead of becoming 0). This dataset covers 73 countries; 20 of them (e.g. Angola, Libya, Nigeria) are
not in the energy-mix dataset, so the two OWID tables only partly overlap.

In [7]:
df_production = pd.read_csv(RAW_DATA_DIR / "fossil-fuel-production" / "fossil-fuel-production.csv")
df_production = df_production.rename(columns={
    "Entity": "country", "Code": "iso3",
    "Coal production": "coal_production_twh",
    "Oil production": "oil_production_twh",
    "Gas production": "gas_production_twh",
})
df_production = df_production[df_production["iso3"].str.match(r"^[A-Z]{3}$", na=False)]
df_production["total_production_twh"] = df_production[["coal_production_twh", "oil_production_twh", "gas_production_twh"]].sum(axis=1, min_count=1)

df_production

,country,iso3,Year,coal_production_twh,oil_production_twh,gas_production_twh,total_production_twh
122,Algeria,DZA,1965,NaN,304.00020,NaN,304.000200
123,Algeria,DZA,1966,NaN,391.26200,NaN,391.262000
124,Algeria,DZA,1967,NaN,449.44415,NaN,449.444150
125,Algeria,DZA,1968,NaN,493.38818,NaN,493.388180
126,Algeria,DZA,1969,NaN,516.78827,NaN,516.788270
...,...,...,...,...,...,...,...
5933,Zimbabwe,ZWE,2021,24.290953,NaN,NaN,24.290953
5934,Zimbabwe,ZWE,2022,29.553642,NaN,NaN,29.553642
5935,Zimbabwe,ZWE,2023,37.412743,NaN,NaN,37.412743
5936,Zimbabwe,ZWE,2024,43.476120,NaN,NaN,43.476120


Litlle test

In [8]:
df_production.loc[(df_production['country'] == 'Netherlands') & (df_production['Year'] >= 2000)]

,country,iso3,Year,coal_production_twh,oil_production_twh,gas_production_twh,total_production_twh
2907,Netherlands,NLD,2000,NaN,NaN,613.611150,613.611150
2908,Netherlands,NLD,2001,NaN,NaN,646.472200,646.472200
2909,Netherlands,NLD,2002,NaN,NaN,635.194460,635.194460
2910,Netherlands,NLD,2003,NaN,NaN,607.166700,607.166700
2911,Netherlands,NLD,2004,NaN,NaN,716.166700,716.166700
2912,Netherlands,NLD,2005,NaN,NaN,653.333300,653.333300
2913,Netherlands,NLD,2006,NaN,NaN,645.027800,645.027800
2914,Netherlands,NLD,2007,NaN,NaN,620.027800,620.027800
2915,Netherlands,NLD,2008,NaN,NaN,709.083400,709.083400
2916,Netherlands,NLD,2009,NaN,NaN,655.000000,655.000000


Merge all df's into 1 df the outer merge gave an issue for Taiwan which is in the OWID datasets but not the world bank data so there needed to be a fix for those rows otherwise we got country nan.

In [9]:
df_master = df_rents.merge(df_energy_mix[["iso3", "Year", "fossil_share_pct"]], on=["iso3", "Year"], how="outer")
df_master = df_master.merge(
    df_production[["iso3", "Year", "coal_production_twh", "oil_production_twh", "gas_production_twh", "total_production_twh"]],
    on=["iso3", "Year"], how="outer",
)

# Recover the country name for rows that only came from an OWID table
# (e.g. Taiwan, which is in OWID but not in the World Bank data).
name_by_iso3 = pd.concat([df_rents[["iso3", "country"]], df_energy_mix[["iso3", "country"]]]).dropna().drop_duplicates("iso3").set_index("iso3")["country"]
df_master["country"] = df_master["country"].fillna(df_master["iso3"].map(name_by_iso3))
df_master = df_master.sort_values(["iso3", "Year"])
df_master["un_session"] = df_master["Year"] - 1945

df_master

,iso3,country,Year,oil_rents_pct_gdp,coal_rents_pct_gdp,gas_rents_pct_gdp,total_fossil_rents_pct_gdp,fossil_share_pct,coal_production_twh,oil_production_twh,gas_production_twh,total_production_twh,un_session
0,ABW,Aruba,1960,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15
1,ABW,Aruba,1961,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16
2,ABW,Aruba,1962,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17
3,ABW,Aruba,1963,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18
4,ABW,Aruba,1964,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,19
...,...,...,...,...,...,...,...,...,...,...,...,...,...
14317,ZWE,Zimbabwe,2021,0.047769,0.290131,0.0,0.3379,NaN,24.290953,NaN,NaN,24.290953,76
14318,ZWE,Zimbabwe,2022,NaN,NaN,NaN,NaN,NaN,29.553642,NaN,NaN,29.553642,77
14319,ZWE,Zimbabwe,2023,NaN,NaN,NaN,NaN,NaN,37.412743,NaN,NaN,37.412743,78
14320,ZWE,Zimbabwe,2024,NaN,NaN,NaN,NaN,NaN,43.476120,NaN,NaN,43.476120,79


Test

In [10]:
df_master.loc[(df_master['country'] == 'Netherlands') & (df_master['Year'] >= 2000)]

,iso3,country,Year,oil_rents_pct_gdp,coal_rents_pct_gdp,gas_rents_pct_gdp,total_fossil_rents_pct_gdp,fossil_share_pct,coal_production_twh,oil_production_twh,gas_production_twh,total_production_twh,un_session
9544,NLD,Netherlands,2000,0.042866,0.0,0.343533,0.386399,98.187130,NaN,NaN,613.611150,613.611150,55
9545,NLD,Netherlands,2001,0.028884,0.0,0.747341,0.776225,98.137650,NaN,NaN,646.472200,646.472200,56
9546,NLD,Netherlands,2002,0.044797,0.0,0.505351,0.550148,98.006590,NaN,NaN,635.194460,635.194460,57
9547,NLD,Netherlands,2003,0.044441,0.0,0.430353,0.474794,98.081184,NaN,NaN,607.166700,607.166700,58
9548,NLD,Netherlands,2004,0.048201,0.0,0.395244,0.443446,97.922410,NaN,NaN,716.166700,716.166700,59
9549,NLD,Netherlands,2005,0.048689,0.0,0.294878,0.343567,97.346090,NaN,NaN,653.333300,653.333300,60
9550,NLD,Netherlands,2006,0.047430,0.0,0.607060,0.654490,97.390390,NaN,NaN,645.027800,645.027800,61
9551,NLD,Netherlands,2007,0.069582,0.0,0.521203,0.590786,97.170900,NaN,NaN,620.027800,620.027800,62
9552,NLD,Netherlands,2008,0.076700,0.0,0.963753,1.040454,96.833430,NaN,NaN,709.083400,709.083400,63
9553,NLD,Netherlands,2009,0.034232,0.0,0.686149,0.720382,96.335045,NaN,NaN,655.000000,655.000000,64


Export to csv

In [11]:
df_rents.to_csv(OUTPUT_DIR / "fossil_rents_by_country_year.csv", index=False)
df_energy_mix.to_csv(OUTPUT_DIR / "energy_mix_fossil_share_by_country_year.csv", index=False)
df_production.to_csv(OUTPUT_DIR / "fossil_fuel_production_by_country_year.csv", index=False)
df_master.to_csv(OUTPUT_DIR / "master_panel_country_year.csv", index=False)

for f in sorted(OUTPUT_DIR.glob("*.csv")):
    print(f"{f.name:45s} {f.stat().st_size/1024:8.1f} KB")

energy_mix_fossil_share_by_country_year.csv      103.4 KB
fossil_fuel_production_by_country_year.csv       178.1 KB
fossil_rents_by_country_year.csv                 685.0 KB
major_fossil_economies.csv                         4.7 KB
master_panel_country_year.csv                    922.8 KB
rq1_correlations.csv                               1.0 KB
rq1_country_summary.csv                           10.8 KB
rq2_coefficients.csv                               1.0 KB
rq2_model_comparison.csv                           0.6 KB
